In [1]:
import warnings
warnings.filterwarnings('ignore')

# Comparando Modelos RF x Regras de Decisão

## Visão Geral

Este painel apresenta uma análise comparativa entre diferentes modelos de seleção de ações. Cada modelo escolhe **10 ações por mês** com base em critérios quantitativos próprios para ajudar investidores a montar portifólios otimizados - com objetivo de gerar retornos maiores do que o Ibovespa. O dashboard consolida todas essas escolhas ao longo do período analisado.

Ou seja, esse é um Helper para portifólios de ações brasileras e deve ser combinado com decisões de alocação ótima entre: ações, renda fixa, cripto, exterior... para proporcionar o máximo rendimento de um investidos amador

## Features Utilizadas

As Features abaixo foram resultado de um filtro com mais Features do mesmo tipo (Indicadores técnicos), que foram descartadas por falta de importância.

Por mais que pareça uma grande quantidade de Features, na prática utilizei um modelo **SelectKBest** a cada Trade para escolher as **5 melhores Features**.

In [2]:
import pandas as pd
import numpy as np

# 1. Organizar os dados em uma lista estruturada
dados = [
    # Média Móvel (Tendência)
    {'Categoria do Indicador': 'Média Móvel (Tendência)', 'Siglas (Features)': 'SMA_20, SMA_50, SMA_200', 'Descrição / Períodos': 'Simple Moving Average (20, 50, 200 dias)'},
    {'Categoria do Indicador': np.nan, 'Siglas (Features)': 'EMA_12, EMA_26, EMA_50', 'Descrição / Períodos': 'Exponential Moving Average (12, 26, 50 dias)'},
    
    # Momentum/Osciladores
    {'Categoria do Indicador': 'Momentum/Osciladores', 'Siglas (Features)': 'RSI_14, MACD, MACD_Hist', 'Descrição / Períodos': 'Relative Strength Index, MACD e seu Histograma'},
    {'Categoria do Indicador': np.nan, 'Siglas (Features)': 'ROC_12', 'Descrição / Períodos': 'Rate of Change (12 dias)'},
    
    # Retorno e Volatilidade
    {'Categoria do Indicador': 'Retorno e Volatilidade', 'Siglas (Features)': 'Ret_1d, Ret_5d, Ret_21d', 'Descrição / Períodos': 'Retornos (diário, semanal, mensal)'},
    {'Categoria do Indicador': np.nan, 'Siglas (Features)': 'Vol_21d, Vol_63d, ATR_14', 'Descrição / Períodos': 'Volatilidade (21/63 dias) e Average True Range'},
    
    # Volume / Alvos
    {'Categoria do Indicador': 'Volume / Alvos', 'Siglas (Features)': 'OBV', 'Descrição / Períodos': 'On-Balance Volume'},
    {'Categoria do Indicador': np.nan, 'Siglas (Features)': 'Target_21, Target_42', 'Descrição / Períodos': 'Variáveis Alvo (Retorno passado após 21 e 42 dias)'}
]

# 2. Criar o DataFrame
df_features = pd.DataFrame(dados)

# 3. Preencher os valores ausentes na coluna 'Categoria do Indicador' para replicar o formato visual (opcional)
df_features['Categoria do Indicador'] = df_features['Categoria do Indicador'].fillna(method='ffill')

# Exibir o DataFrame
df_features.set_index('Categoria do Indicador')

,Siglas (Features),Descrição / Períodos
Categoria do Indicador,,
Média Móvel (Tendência),"SMA_20, SMA_50, SMA_200","Simple Moving Average (20, 50, 200 dias)"
Média Móvel (Tendência),"EMA_12, EMA_26, EMA_50","Exponential Moving Average (12, 26, 50 dias)"
Momentum/Osciladores,"RSI_14, MACD, MACD_Hist","Relative Strength Index, MACD e seu Histograma"
Momentum/Osciladores,ROC_12,Rate of Change (12 dias)
Retorno e Volatilidade,"Ret_1d, Ret_5d, Ret_21d","Retornos (diário, semanal, mensal)"
Retorno e Volatilidade,"Vol_21d, Vol_63d, ATR_14",Volatilidade (21/63 dias) e Average True Range
Volume / Alvos,OBV,On-Balance Volume
Volume / Alvos,"Target_21, Target_42",Variáveis Alvo (Retorno passado após 21 e 42 d...


## Do Número e Frequência de Trades

A métrica escolhida para o teste foi utilizar um portifólio de um mês, escolhendo-se até 10 ativos. Portanto, temos um número de 10 Trades por mês. Essa métrica é escolhida como parâmetro do modelo, então podemos altera-la livremente.

## Do Universo de Ativos

Ao todo, foram utilizadas 92 ações, escolhidas independentemente por modelo. Assim abaixo há a tabela de ações escolhidas por modelos ao longo de toda a trajetória do período de teste.

In [16]:
SETORES = {
    "BANCOS": [
        "ITUB4.SA", "BBDC4.SA", "BBAS3.SA", "SANB11.SA", "BPAC11.SA",
        "CXSE3.SA", "BRAP4.SA", "BRSR6.SA", "CRFB3.SA", "PSSA3.SA", "PINE4.SA"
    ],

    "ENERGIA": [
        "PETR4.SA", "PRIO3.SA", "OIBR4.SA", "ELET3.SA", "CMIG4.SA",
        "CPFE3.SA", "EGIE3.SA", "ENGI11.SA", "GEMA3.SA", "LIGHT3.SA",
        "TRPL4.SA", "EQTL3.SA"
    ],

    "MINERAÇÃO": [
        "VALE3.SA", "CSNA3.SA", "USIM5.SA", "GGBR4.SA"
    ],

    "VAREJO": [
        "MGLU3.SA", "LREN3.SA", "ABEV3.SA", "RENT3.SA",
        "MOVI3.SA", "VVAR3.SA", "PCAR3.SA", "TRIS3.SA"
    ],

    "CONSUMO": [
        "WEGE3.SA", "JBSS3.SA", "MSFT34.SA", "HYPE3.SA", "SLCE3.SA",
        "PETZ3.SA", "ARZZ3.SA", "TFCO4.SA", "BRML3.SA"
    ],

    "TRANSPORTE": [
        "RAIL3.SA", "CCRO3.SA", "LOGB3.SA", "ARZZ3.SA", "EMAE4.SA", "ATUS3.SA"
    ],

    "CONSTRUÇÃO": [
        "MRVE3.SA", "TEND3.SA", "PLPL3.SA", "GFSA3.SA", "TRAD3.SA"
    ],

    "IMÓVEIS": [
        "VLID3.SA", "BRIV3.SA", "CYRE3.SA", "EVEN3.SA", "HBOR3.SA"
    ],

    "COMUNICAÇÃO": [
        "VIVT3.SA", "TIMS3.SA", "OIBR3.SA"
    ],

    "PAPEL E CELULOSE": [
        "SUZB3.SA", "SBSP3.SA", "KLABIN11.SA", "FIBR3.SA"
    ],

    "QUÍMICA/HIGIENE": [
        "TOTS3.SA", "BRPR3.SA", "CLSA3.SA"
    ],

    "ALIMENTOS": [
        "MBLY3.SA", "BRF3.SA", "SEQL3.SA", "ASAI3.SA"
    ],

    "TECNOLOGIA": [
        "TOTS3.SA", "NTCO3.SA", "BRQT3.SA", "DIRR3.SA", "TRPL4.SA"
    ],

    "AVIAÇÃO": [
        "EMBR3.SA", "AZUL4.SA", "GOLL4.SA"
    ],

    "SEGUROS": [
        "PSSA3.SA", "SULB3.SA", "SGUP3.SA"
    ],

    "FINANCEIRAS": [
        "B3SA3.SA", "MOVI3.SA", "RBRR3.SA", "RDOR3.SA"
    ],

    "AGRONEGÓCIO": [
        "AGRO3.SA", "AERI3.SA", "POSI3.SA"
    ]
}
TICKERS_EXPANDIDA = [
        # BANCOS (11)
        "ITUB4.SA",  # Itaú Unibanco
        "BBDC4.SA",  # Bradesco
        "BBAS3.SA",  # Banco do Brasil
        "SANB11.SA", # Santander
        "BPAC11.SA", # Banco do Brasil PN
        "CXSE3.SA",  # Caixa Seguridade
        "BRAP4.SA",  # Bradespar
        "BRSR6.SA",  # Banco do Brasil ON
        "CRFB3.SA",  # Carrefour Brasil
        "PSSA3.SA",  # Porto Seguro
        "PINE4.SA",  # Banco Pine
        
        # ENERGIA (12)
        "PETR4.SA",  # Petrobras PN
        "PRIO3.SA",  # Petrorio
        "OIBR4.SA",  # Oi PN
        "ELET3.SA",  # Eletrobras ON
        "CMIG4.SA",  # Cemig PN
        "CPFE3.SA",  # CPFL Energia
        "EGIE3.SA",  # EDP Energias
        "ENGI11.SA", # Engie Brasil
        "GEMA3.SA",  # Gerdau Metalúrgica
        "LIGHT3.SA", # Light
        "TRPL4.SA",  # Transmissão Paulista
        "EQTL3.SA",  # Equatorial Energia
        
        # MINERAÇÃO (4)
        "VALE3.SA",  # Vale
        "CSNA3.SA",  # Companhia Siderúrgica
        "USIM5.SA",  # Usiminas
        "GGBR4.SA",  # Gerdau PN
        
        # VAREJO (8)
        "MGLU3.SA",  # Magazine Luiza
        "LREN3.SA",  # Lojas Renner
        "ABEV3.SA",  # Ambev
        "RENT3.SA",  # Localiza
        "MOVI3.SA",  # Movida
        "VVAR3.SA",  # Via Varejo
        "PCAR3.SA",  # Impar
        "TRIS3.SA",  # Triscila
        
        # CONSUMO (9)
        "WEGE3.SA",  # WEG
        "JBSS3.SA",  # JBS
        "MSFT34.SA", # Microsoft (ADR)
        "HYPE3.SA",  # Hypera
        "SLCE3.SA",  # SLC Agrícola
        "PETZ3.SA",  # Petz
        "ARZZ3.SA",  # Arezzo
        "TFCO4.SA",  # Telefônico Brasil
        "BRML3.SA",  # Brasil Malha Logística
        
        # TRANSPORTE (6)
        "RAIL3.SA",  # Rumo
        "CCRO3.SA",  # CCR
        "LOGB3.SA",  # Loggi
        "ARZZ3.SA",  # Arezzo (calçados)
        "EMAE4.SA",  # Emae
        "ATUS3.SA",  # Atus
        
        # CONSTRUÇÃO (5)
        "MRVE3.SA",  # MRV Engenharia
        "TEND3.SA",  # Construtora Tenda
        "PLPL3.SA",  # Plano & Plano
        "GFSA3.SA",  # Gafisa
        "TRAD3.SA",  # Tradição
        
        # IMÓVEIS (5)
        "VLID3.SA",  # Validada Imóveis
        "BRIV3.SA",  # BR Imobiliário
        "CYRE3.SA",  # Cyrela
        "EVEN3.SA",  # Even
        "HBOR3.SA",  # Helbor
        
        # COMUNICAÇÃO (3)
        "VIVT3.SA",  # Vivo
        "TIMS3.SA",  # Tim
        "OIBR3.SA",  # Oi ON
        
        # PAPEL E CELULOSE (4)
        "SUZB3.SA",  # Suzano
        "SBSP3.SA",  # Sabesp
        "KLABIN11.SA", # Klabin
        "FIBR3.SA",  # Fibria
        
        # QUÍMICA/HIGIENE (3)
        "TOTS3.SA",  # Totvs
        "BRPR3.SA",  # Brasilfops
        "CLSA3.SA",  # Classa
        
        # ALIMENTOS (4)
        "MBLY3.SA",  # Marfrig
        "BRF3.SA",   # BRF
        "SEQL3.SA",  # Sequoia
        "ASAI3.SA",  # Assaí
        
        # TECNOLOGIA (5)
        "TOTS3.SA",  # Totvs
        "NTCO3.SA",  # Natura
        "BRQT3.SA",  # Brq Digital
        "DIRR3.SA",  # Direcional Engenharia
        "TRPL4.SA",  # Transmissão Paulista
        
        # AVIAÇÃO (3)
        "EMBR3.SA",  # Embraer
        "AZUL4.SA",  # Azul
        "GOLL4.SA",  # Gol
        
        # SEGUROS (3)
        "PSSA3.SA",  # Porto Seguro
        "SULB3.SA",  # Sulamerica
        "SGUP3.SA",  # Seguradoras Unidas
        
        # FINANCEIRAS (4)
        "B3SA3.SA",  # B3
        "MOVI3.SA",  # Movida
        "RBRR3.SA",  # Rede Brasil Real
        "RDOR3.SA",  # Rede D'Or
        
        # AGRONEGÓCIO (3)
        "AGRO3.SA",  # Agrogalaxy
            "AERI3.SA",  # Aerea Invest
        "POSI3.SA",  # Positivo
        ]


from utils.visualizacao import main_visualizacao
caminhos = ["portfolios_mensais_RF_2023-01-01_2025-11-01.csv",
            "portfolios_mensais_retornoultimos5m_2023-01-01_2025-11-01.csv",
            "portfolios_mensais_retornoultimomes_2023-01-01_2025-11-01.csv",
            "portfolios_mensais_retornohistorico_2023-01-01_2025-11-01.csv",
            "portfolios_mensais_menorvariancia_2023-01-01_2025-11-01.csv",
            "portfolios_mensais_media3m_2023-01-01_2025-11-01.csv"]
df_final, df_metricas, tickers = main_visualizacao(caminhos=caminhos)

# Monta MultiIndex (setor, ação)
multi_idx = []
for setor, acoes in SETORES.items():
    for acao in acoes:
        multi_idx.append((setor, acao))

# Cria DataFrame booleano
df = pd.DataFrame(index=pd.MultiIndex.from_tuples(multi_idx, names=['Setor', 'Ação']))
for modelo in tickers:
    df[modelo] = [acao in tickers[modelo] for setor, acao in multi_idx]

df

[*********************100%***********************]  60 of 60 completed
[*********************100%***********************]  49 of 49 completed
[*********************100%***********************]  62 of 62 completed
[*********************100%***********************]  17 of 17 completed
[*********************100%***********************]  8 of 8 completed
[*********************100%***********************]  60 of 60 completed
[*********************100%***********************]  1 of 1 completed


RF  retornoultimos5m  retornoultimomes  \
Setor       Ação                                                   
BANCOS      ITUB4.SA   False             False             False   
            BBDC4.SA   False              True              True   
            BBAS3.SA    True              True              True   
            SANB11.SA   True             False              True   
            BPAC11.SA   True             False              True   
...                      ...               ...               ...   
FINANCEIRAS RBRR3.SA   False             False             False   
            RDOR3.SA    True              True              True   
AGRONEGÓCIO AGRO3.SA    True              True              True   
            AERI3.SA    True              True              True   
            POSI3.SA    True              True              True   

                       retornohistorico  menorvariancia  media3m  
Setor       Ação                                                  
BANCOS      ITUB4.SA              False            True    False  
            BBDC4.SA              False            True     True  
            BBAS3.SA              False            True     True  
            SANB11.SA             False            True    False  
            BPAC11.SA              True            True     True  
...                                 ...             ...      ...  
FINANCEIRAS RBRR3.SA              False           False    False  
            RDOR3.SA              False           False     True  
AGRONEGÓCIO AGRO3.SA              False           False     True  
            AERI3.SA              False           False     True  
            POSI3.SA               True           False     True  

[92 rows x 6 columns]

## Período de Teste

As datas inciais e finais do **backtest**
- **Data inicial:** 01/01/2023  
- **Data final:** 01/11/2025


Perceba que não há data inicial, já que para cada ação, simplesmente peguei o máximo de dados que eu teria acesso.

## Portifólios

Para o determinado período de teste, essas foram as proporções escolhidas via **Máximo Sharpe Ponderado**

*Ponderado pelo erro observado para aquela ação até a data de hoje*

In [17]:
portifolios_RF = pd.read_csv('portfolios_mensais_RF_2023-01-01_2025-11-01.csv')

portifolios_RF = portifolios_RF.set_index('Mes')

portifolios_RF[portifolios_RF.columns.drop(['Data','Modelo'])]

,EMAE4.SA,GFSA3.SA,AGRO3.SA,BRAP4.SA,BRPR3.SA,MGLU3.SA,OIBR3.SA,CSNA3.SA,PCAR3.SA,BBAS3.SA,...,TOTS3.SA,RAIL3.SA,EQTL3.SA,SEQL3.SA,B3SA3.SA,ABEV3.SA,ELET3.SA,RENT3.SA,ASAI3.SA,TFCO4.SA
Mes,,,,,,,,,,,,,,,,,,,,,
2023-01,0.400000,0.116800,0.129361,0.168566,0.185272,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-02,NaN,NaN,NaN,NaN,NaN,0.171471,0.072779,0.241962,0.104214,0.296944,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.047559,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-05,0.220690,NaN,NaN,NaN,0.001643,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-06,NaN,NaN,NaN,NaN,0.005880,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-07,NaN,0.017949,NaN,NaN,0.008572,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Métricas de Portifólio

As métricas do retorno do portifólio estão presentes aqui.

In [18]:
df_metricas_ = df_metricas.loc[df_metricas.index[:5]]
df_metricas_ = df_metricas_.round(2)  # opcional, para deixar bonito
df_metricas_ = df_metricas_.applymap(lambda x: f"{x*100:.2f}%")
df_metricas_

,RF,retornoultimos5m,retornoultimomes,retornohistorico,menorvariancia,media3m,IBOV
Retorno acumulado,29.42%,31.21%,14.27%,230.92%,97.60%,90.69%,44.83%
Retorno anualizado,12.53%,13.74%,9.17%,70.75%,30.46%,28.23%,15.24%
Retorno médio diário,0.05%,0.05%,0.03%,0.21%,0.11%,0.10%,0.06%
Volatilidade diária,1.47%,1.60%,1.83%,3.67%,1.43%,1.29%,0.94%
Volatilidade anualizada,23.37%,25.37%,29.00%,58.27%,22.69%,20.44%,14.96%


## DataFrame Final

Aqui há o dataframe final indexados com o início sendo 100. Esse DF representa o resultado de cada um dos portifólios ao longo do tempo

In [26]:
df_final

,RF,retornoultimos5m,retornoultimomes,retornohistorico,menorvariancia,media3m,IBOV
Date,,,,,,,
2023-01-02,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
2023-01-03,105.323117,98.021354,120.054320,96.323298,95.723490,104.572109,97.922464
2023-01-04,103.937044,98.262151,117.857255,96.934831,97.273648,103.781531,99.020456
2023-01-05,110.593829,99.961325,140.657184,98.371293,101.393515,111.064524,101.073550
2023-01-06,107.911481,99.285948,127.088538,98.936275,103.052196,108.772024,102.312552
...,...,...,...,...,...,...,...
2025-11-03,126.761646,128.415403,112.179019,324.616013,192.022115,182.173907,141.436038
2025-11-04,127.842583,128.593421,113.152927,326.435077,193.070925,182.976679,141.671054
2025-11-05,130.171627,131.184338,115.376231,331.403663,197.215406,186.850480,144.105813


## Gráfico do Retorno

O graficó do retorno dos RF comparado ao ibovespa e aos melhores modelos

In [34]:
from utils.visualizacao import grafico_retorno, grafico_alpha

df_plot1 = df_final[['RF',	'menorvariancia','media3m', 'IBOV']]
grafico_retorno(df_plot1)

alt.Chart(...)

## Gráfico do Alpha

Perceba que o alpha acumulado dos melhores modelos contra o RF é especialmente relevante. Perceba que esses outros modelos são bem mais simples, e no caso do "menorvariancia" ele simplesmente esta chutando o retorno como sendo zero. Isso significa que nosso random forest perceu para o absolutamente nada, literalmente

In [35]:
grafico_alpha(df_plot1)

alt.LayerChart(...)